# Study evidence

The default renders invented aggregates. Development mode renders the saved Slice 5 and Slice 6 results. Final mode renders the seven verified Slice 8 aggregate files. Set the mode and input paths and digests as described in `docs/study-evidence.md`.

All metrics use the canonical ten labels. The transformer sample SD measures variability across three seeds on the same rows within the selected scope. Recall is unavailable for zero true support. Precision is unavailable for zero predicted count.

In [ ]:
import os
from pathlib import Path

from disaster_triage.evidence import render_evidence

cwd = Path.cwd()
default_root = cwd.parent if cwd.name == "notebooks" else cwd
root = Path(os.environ.get("TRIAGE_EVIDENCE_ROOT", default_root))
mode = os.environ.get("TRIAGE_EVIDENCE_MODE", "synthetic")
output_id = os.environ.get("TRIAGE_EVIDENCE_OUTPUT_ID", "study-evidence-synthetic")
result = render_evidence(
    root, output_id, mode=mode,
    selection_path=os.environ.get("TRIAGE_EVIDENCE_SELECTION"),
    selection_sha256=os.environ.get("TRIAGE_EVIDENCE_SELECTION_SHA256"),
    summary_path=os.environ.get("TRIAGE_EVIDENCE_SUMMARY"),
    summary_sha256=os.environ.get("TRIAGE_EVIDENCE_SUMMARY_SHA256"),
    final_manifest_path=os.environ.get("TRIAGE_EVIDENCE_FINAL_MANIFEST"),
    final_manifest_sha256=os.environ.get("TRIAGE_EVIDENCE_FINAL_MANIFEST_SHA256"),
)
output = Path(result["output"])
print(f"{mode} · manifest SHA-256 {result['manifest_sha256']}")

In [ ]:
import csv
from html import escape

from IPython.display import HTML, Image, display

def show_table(filename, title, omit=(), collapsed=False):
    with (output / filename).open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        columns = [column for column in reader.fieldnames if column not in omit]
        header = "".join(f"<th>{escape(column)}</th>" for column in columns)
        body = "".join(
            "<tr>" + "".join(f"<td>{escape(row[column])}</td>" for column in columns) + "</tr>"
            for row in reader
        )
    table = f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
    markup = f"<h3>{escape(title)}</h3>{table}"
    if collapsed:
        markup = f"<details><summary>{escape(title)}</summary>{table}</details>"
    display(HTML(markup))

if mode == "final":
    show_table("event-support.csv", "Event support and common-label comparison", collapsed=True)
    show_table("calibration.csv", "Calibration before and after temperature scaling")
    show_table("review.csv", "Frozen review scenarios", collapsed=True)
else:
    show_table("candidate-scores.csv", "Candidate scores", omit=("run_id", "config_sha256"))
show_table("model-comparison.csv", "Model comparison")
show_table("class-metrics.csv", "Class metrics", collapsed=True)

In [ ]:
for figure in sorted(output.glob("figure-*.png")):
    display(Image(filename=str(figure), width=720))